# PET Bottle YOLOv8 Training
Train a custom YOLOv8s model on the PET Bottles v2 dataset from Roboflow.

**Run this in Google Colab with GPU runtime:**
- Runtime → Change runtime type → T4 GPU
- Expected training time: ~30-45 min on T4

## Step 1: Install Dependencies

In [ ]:
!pip install ultralytics roboflow -q

## Step 2: Download Dataset from Roboflow
This downloads your PET Bottles v2 dataset with proper train/val/test split.

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="7uuXglhdXfyNxjhmZB0b")
project = rf.workspace("street-photos").project("pet-bottles-v2")

# Download version 3 in YOLOv8 format
dataset = project.version(3).download("yolov8")
print(f"Dataset location: {dataset.location}")

## Step 3: Fix Dataset Split
The dataset has all 1468 images in train with 0 valid/test.
We need to split it 80/15/5 for proper training.

In [ ]:
import os, shutil, random, glob

dataset_dir = dataset.location
train_img = os.path.join(dataset_dir, "train", "images")
train_lbl = os.path.join(dataset_dir, "train", "labels")
valid_img = os.path.join(dataset_dir, "valid", "images")
valid_lbl = os.path.join(dataset_dir, "valid", "labels")
test_img = os.path.join(dataset_dir, "test", "images")
test_lbl = os.path.join(dataset_dir, "test", "labels")

# Create dirs
for d in [valid_img, valid_lbl, test_img, test_lbl]:
    os.makedirs(d, exist_ok=True)

# Check if split is needed
existing_valid = len(glob.glob(os.path.join(valid_img, "*")))
if existing_valid > 0:
    print(f"Valid set already has {existing_valid} images — skipping split")
else:
    # Get all training images
    images = sorted(glob.glob(os.path.join(train_img, "*")))
    random.seed(42)
    random.shuffle(images)
    
    n = len(images)
    n_val = int(n * 0.15)
    n_test = int(n * 0.05)
    
    val_images = images[:n_val]
    test_images = images[n_val:n_val + n_test]
    
    def move_to(img_list, dst_img, dst_lbl):
        for img_path in img_list:
            fname = os.path.basename(img_path)
            lbl_name = os.path.splitext(fname)[0] + ".txt"
            lbl_path = os.path.join(train_lbl, lbl_name)
            shutil.move(img_path, os.path.join(dst_img, fname))
            if os.path.exists(lbl_path):
                shutil.move(lbl_path, os.path.join(dst_lbl, lbl_name))
    
    move_to(val_images, valid_img, valid_lbl)
    move_to(test_images, test_img, test_lbl)
    
    print(f"Split complete:")
    print(f"  Train: {len(glob.glob(os.path.join(train_img, '*')))} images")
    print(f"  Valid: {len(glob.glob(os.path.join(valid_img, '*')))} images")
    print(f"  Test:  {len(glob.glob(os.path.join(test_img, '*')))} images")

## Step 4: Check data.yaml

In [ ]:
import yaml

data_yaml = os.path.join(dataset_dir, "data.yaml")
with open(data_yaml) as f:
    data = yaml.safe_load(f)

print("Classes:", data.get('names', data.get('nc', '?')))
print("NC:", data.get('nc', '?'))
print("Train:", data.get('train'))
print("Val:", data.get('val'))
print("Test:", data.get('test', 'not set'))

## Step 5: Train YOLOv8s
Training with proper settings:
- **150 epochs** (was probably ~50 before)
- **Image size 640** (matches deployment)
- **Augmentation** enabled (mosaic, flip, hsv, scale)
- **Patience 30** (early stopping if no improvement)

In [ ]:
from ultralytics import YOLO

# Load YOLOv8s pretrained on COCO
model = YOLO("yolov8s.pt")

# Train on PET bottle dataset
results = model.train(
    data=data_yaml,
    epochs=150,
    imgsz=640,
    batch=16,
    patience=30,         # early stop if no improvement for 30 epochs
    save=True,
    save_period=25,      # save checkpoint every 25 epochs
    device=0,            # GPU
    workers=4,
    # Augmentation
    augment=True,
    mosaic=1.0,          # mosaic augmentation
    flipud=0.1,          # vertical flip 10%
    fliplr=0.5,          # horizontal flip 50%
    hsv_h=0.015,         # hue shift
    hsv_s=0.7,           # saturation shift
    hsv_v=0.4,           # brightness shift
    scale=0.5,           # scale augmentation
    translate=0.1,       # translation
    degrees=10,          # rotation ±10°
    mixup=0.1,           # mixup augmentation
    # Optimizer
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,            # final LR = lr0 * lrf
    weight_decay=0.0005,
    warmup_epochs=5,
    # Names
    name="petbottle_v8s_v2",
)

## Step 6: Evaluate Results

In [ ]:
import pandas as pd
from IPython.display import display, Image as IPImage

# Training results
results_csv = os.path.join(results.save_dir, "results.csv")
if os.path.exists(results_csv):
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()
    print("\n=== Final Metrics ===")
    last = df.iloc[-1]
    print(f"  mAP@50:     {last.get('metrics/mAP50(B)', 'N/A'):.4f}")
    print(f"  mAP@50-95:  {last.get('metrics/mAP50-95(B)', 'N/A'):.4f}")
    print(f"  Precision:  {last.get('metrics/precision(B)', 'N/A'):.4f}")
    print(f"  Recall:     {last.get('metrics/recall(B)', 'N/A'):.4f}")
    print(f"  Epochs:     {len(df)}")

# Show training curves
results_img = os.path.join(results.save_dir, "results.png")
if os.path.exists(results_img):
    display(IPImage(filename=results_img, width=800))

# Confusion matrix
cm_img = os.path.join(results.save_dir, "confusion_matrix.png")
if os.path.exists(cm_img):
    display(IPImage(filename=cm_img, width=600))

## Step 7: Validate on Test Set

In [ ]:
# Load best weights and validate
best_pt = os.path.join(results.save_dir, "weights", "best.pt")
best_model = YOLO(best_pt)

# Run validation on test set
test_results = best_model.val(data=data_yaml, split="test")
print(f"\n=== Test Set Results ===")
print(f"  mAP@50:    {test_results.box.map50:.4f}")
print(f"  mAP@50-95: {test_results.box.map:.4f}")
print(f"  Precision: {test_results.box.mp:.4f}")
print(f"  Recall:    {test_results.box.mr:.4f}")

## Step 8: Export to ONNX
Export for Hailo HEF conversion.

In [ ]:
# Export to ONNX (for Hailo DFC conversion)
best_model.export(
    format="onnx",
    imgsz=416,       # match current Hailo input size
    simplify=True,
    opset=11,
)

onnx_path = best_pt.replace(".pt", ".onnx")
print(f"\nONNX exported: {onnx_path}")
print(f"Size: {os.path.getsize(onnx_path) / 1e6:.1f} MB")

## Step 9: Download Trained Weights
Download `best.pt` and `best.onnx` to your local machine.

In [ ]:
# Copy files to easy download location
import shutil

download_dir = "/content/download"
os.makedirs(download_dir, exist_ok=True)

shutil.copy(best_pt, os.path.join(download_dir, "petbottle-yolov8s-v2.pt"))
shutil.copy(onnx_path, os.path.join(download_dir, "petbottle-yolov8s-v2.onnx"))

print("\n=== Download these files ===")
print(f"  1. {download_dir}/petbottle-yolov8s-v2.pt   (PyTorch weights)")
print(f"  2. {download_dir}/petbottle-yolov8s-v2.onnx (for Hailo HEF conversion)")
print("\nIn Colab: Files panel (left) → download/  → right-click → Download")

## Step 10: Convert ONNX to HEF (Hailo)
This step requires the **Hailo Dataflow Compiler (DFC)** which runs on x86 Linux.

Option A: Install DFC on an x86 Linux machine:
```bash
hailo compiler petbottle-yolov8s-v2.onnx --hw-arch hailo8
```

Option B: Use Hailo Model Zoo:
```bash
pip install hailo_model_zoo
hailomz compile yolov8s --ckpt petbottle-yolov8s-v2.onnx --hw-arch hailo8 --classes 1
```

Then copy the `.hef` file to the Pi:
```bash
scp petbottle-yolov8s-v2.hef set-admin@<pi-ip>:/home/set-admin/testing/
```

## Comparison: What Changed

| Setting | Old Training | New Training |
|---------|-------------|-------------|
| Validation split | 0% | 15% |
| Augmentation | None | Mosaic + flip + HSV + scale + rotate + mixup |
| Epochs | ~50? | 150 (with early stopping) |
| Optimizer | SGD (default) | AdamW |
| Image size | 416 | 640 (better detail) |
| Warmup | Default | 5 epochs |